In [2]:

print("Pyproj version:", pyproj.__version__)
print("Data directory:", pyproj.datadir.get_data_dir())

Pyproj version: 3.7.2
Data directory: C:\Users\Administrator\.local\share\mamba\envs\mgr\Library\share\proj


In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine, text
import pyproj

In [16]:
TARGET_CRS = 3035
layers = {
    "forecastCBDG": ("../data/processed/forecast/prognoz_osiad_19_buff30_summarizeWithin.shp"),
    "floodingCBDG": ("../data/raw/cbdg_hydro_podtopienia_2022_04_01/cbdg_hydro_podtopienia_2022_04_01.shp"),
    "clc18": ("../data/raw/clc18_PL_1/clc18_PL.shp")
}

mask = ("../data/processed/extent/extent_fc.shp")

In [11]:
maska = gpd.read_file(mask)

In [4]:
db_user = "postgres"
db_password = "postgres"
db_host = "localhost"
db_port = "5432"
db_name = "niecki"

engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

In [12]:
def load_layer(table, path):
    gdf = gpd.read_file(path)
    gdf = gdf.to_crs(TARGET_CRS)
    gdf = gpd.clip(gdf, maska)
    gdf.columns = [c.lower() for c in gdf.columns]   # COType -> cotype
    gdf["geometry"] = gdf.geometry.make_valid()
    gdf = gdf.rename_geometry("geom")
    gdf.to_postgis(table, engine, if_exists="replace", index=False)
    with engine.begin() as con:
        con.execute(text(f"CREATE INDEX IF NOT EXISTS {table}_geom_gix ON {table} USING GIST (geom)"))
        con.execute(text(f"ANALYZE {table}"))

In [17]:
for t, p in layers.items():
    load_layer(t, p)

In [ ]:
#zmiana geometrii z 3d na 2d
engine.begin(text(f"ALTER TABLE clc2018 ALTER COLUMN geom TYPE geometry(MultiPolygon, 3035) USING ST_Multi(ST_Force2D(geom));"))